In [15]:
# overlay_utils.py
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()

def apply_overlay(base_image_path: str, overlay_landscape_path: str,
                  overlay_portrait_path: str, output_path: str) -> None:
    """
    Applies the correct overlay based on the base image's orientation.

    - Landscape base (w >= h) → uses overlay_landscape_path
    - Portrait base  (h > w)  → uses overlay_portrait_path

    Rules for combining:
      - Overlay bigger than base → overlay scaled down to fit (never cropped).
      - Base bigger than overlay → base center-cropped to overlay's size.
    """
    base = Image.open(base_image_path).convert("RGBA")
    base = ImageOps.exif_transpose(base)   # respect phone rotation

    bw, bh = base.size
    overlay_path = overlay_landscape_path if bw >= bh else overlay_portrait_path
    overlay = Image.open(overlay_path).convert("RGBA")

    ow, oh = overlay.size

    if ow > bw or oh > bh:
        # Overlay bigger → scale it down to fit inside base
        overlay.thumbnail(base.size, Image.Resampling.LANCZOS)
        ow, oh = overlay.size
        base = center_crop(base, ow, oh)
    else:
        # Base bigger (or equal) → center-crop base to overlay's size
        base = center_crop(base, ow, oh)

    x = (base.width - ow) // 2
    y = (base.height - oh) // 2
    base.paste(overlay, (x, y), overlay)

    base.convert("RGB").save(output_path, quality=95)


def center_crop(img: Image.Image, target_w: int, target_h: int) -> Image.Image:
    """Center-crop an image to the target width/height."""
    w, h = img.size
    left = (w - target_w) // 2
    top = (h - target_h) // 2
    right = left + target_w
    bottom = top + target_h
    return img.crop((left, top, right, bottom))

In [14]:
# ABC_ABC.jpg
"""
A: L for landscape, P for portrait, S for square
B: H for high resolution, L for low resolution
C: O for overlay, I for input
"""

sample_image = "Sample Inputs/PHI/PHI_1.JPG"
overlay_image_landscape = "Sample Overlays/landscape_lowres.png"
overlay_image_portrait = "Sample Overlays/portrait_lowres.png"

apply_overlay(
    base_image_path=sample_image,
    overlay_landscape_path=overlay_image_landscape,
    overlay_portrait_path=overlay_image_portrait,
    output_path="Outputs/LLO_PLO_PHI.jpg"
)